# Stitch Integration

This notebook demonstrates the **Stitch** stage of the Foreign Whispers dubbing pipeline.
It performs final video assembly: combining the original video with dubbed TTS audio and
TTS-scheduled WebVTT captions via ffmpeg. The stitch uses audio-only remux
(no re-encoding), preserving original video quality.

**Prerequisites:**
- The Docker stack must be running (`docker compose --profile nvidia up -d`).
- The API should be accessible at `http://localhost:8080`.
- Prior pipeline stages (download, transcribe, translate, TTS) must have completed for the target video.

## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)

# Load .env (LOGFIRE_TOKEN, HF_TOKEN, etc.)
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from foreign_whispers.client import FWClient, BASELINE

fw = FWClient("http://localhost:8080")
fw.healthz()

# Optional: Logfire tracing (no-op shim if unavailable)
try:
    import logfire
    logfire.configure(service_name="foreign-whispers-stitch")
    print("Logfire tracing enabled.")
except Exception:
    class _NoopSpan:
        def __enter__(self): return self
        def __exit__(self, *a): pass
    class _noop:
        @staticmethod
        def span(name, **kw): return _NoopSpan()
        @staticmethod
        def info(*a, **kw): pass
    logfire = _noop()
    print("Logfire not configured — using no-op shim.")

Project root: /root/foreign-whispers


Logfire not configured — using no-op shim.


## Stitch Video

Call the API to combine the original video with the dubbed TTS audio track.
The stitch endpoint replaces the audio stream via ffmpeg remux (no video re-encoding)
and generates WebVTT captions synchronized to the assembled TTS timeline.

In [2]:
video_id = "GYQ5yGV_-Oc"
config = "c-e5604bc"

with logfire.span("stitch", video_id=video_id, config=config):
    result = fw.stitch(video_id, config=config)

print(f"Video ID:   {result['video_id']}")
print(f"Video path: {result['video_path']}")
print(f"Config:     {result['config']}")

Video ID:   GYQ5yGV_-Oc
Video path: /app/pipeline_data/api/dubbed_videos/c-e5604bc/Strait of Hormuz disruption threatens to shake global economy.mp4
Config:     c-e5604bc


## Inspect Output Artifacts

The stitch stage produces files in two directories:

- `pipeline_data/api/dubbed_videos/{config}/` — final dubbed MP4 files
- `pipeline_data/api/dubbed_captions/` — target-language VTT caption files

In [3]:
dubbed_videos_dir = PROJECT_ROOT / "pipeline_data" / "api" / "dubbed_videos"
dubbed_captions_dir = PROJECT_ROOT / "pipeline_data" / "api" / "dubbed_captions"

TITLE = "Strait of Hormuz disruption threatens to shake global economy"
CONFIG = "c-e5604bc"

final_video = dubbed_videos_dir / CONFIG / f"{TITLE}.mp4"
final_caption_candidates = [
    dubbed_captions_dir / CONFIG / f"{TITLE}.vtt",
    dubbed_captions_dir / f"{TITLE}.vtt",
]

print("Final dubbed video:")
print(" ", final_video)
print(" exists:", final_video.exists())
if final_video.exists():
    print(f" size_mb: {final_video.stat().st_size / (1024 * 1024):.1f}")

print("\nFinal dubbed captions:")
for p in final_caption_candidates:
    if p.exists():
        print(" ", p)
        print(" exists:", True)
        print(f" size_kb: {p.stat().st_size / 1024:.1f}")

Final dubbed video:
  /root/foreign-whispers/pipeline_data/api/dubbed_videos/c-e5604bc/Strait of Hormuz disruption threatens to shake global economy.mp4
 exists: True
 size_mb: 37.3

Final dubbed captions:
  /root/foreign-whispers/pipeline_data/api/dubbed_captions/Strait of Hormuz disruption threatens to shake global economy.vtt
 exists: True
 size_kb: 12.1


## View Generated Captions

The stitch stage generates WebVTT captions synchronized to the assembled TTS timeline.
Each cue shows only the current translated subtitle, wrapped to at most two lines.

In [4]:
TITLE = "Strait of Hormuz disruption threatens to shake global economy"
config = "c-e5604bc"

vtt_candidates = [
    dubbed_captions_dir / config / f"{TITLE}.vtt",
    dubbed_captions_dir / f"{TITLE}.vtt",
]

vtt_files = [p for p in vtt_candidates if p.exists()]

if not vtt_files:
    vtt_files = list(dubbed_captions_dir.rglob(f"*{TITLE}*.vtt"))

if vtt_files:
    vtt_path = vtt_files[0]
    print(f"Caption file: {vtt_path}\n")
    lines = vtt_path.read_text().splitlines()
    for line in lines[:30]:
        print(line)
    if len(lines) > 30:
        print(f"\n... ({len(lines) - 30} more lines)")
    print()
    print("The VTT file is synchronized to the TTS assembly timeline.")
    print("Each cue shows only the current subtitle, wrapped to at most two lines.")
else:
    print("No VTT files found. Run the stitch step first.")

Caption file: /root/foreign-whispers/pipeline_data/api/dubbed_captions/Strait of Hormuz disruption threatens to shake global economy.vtt

WEBVTT

1
00:00:02.320 --> 00:00:06.120
60 minutos horas extra.

2
00:00:06.120 --> 00:00:07.640
¿Cuál es el peor escenario del caso?

3
00:00:07.640 --> 00:00:09.880
Te preocupa que sea

4
00:00:09.880 --> 00:00:12.040
cerrado durante semanas y semanas y
semanas

5
00:00:12.040 --> 00:00:15.399
y empiezas a ver el mundo

6
00:00:15.399 --> 00:00:17.880
la economía realmente impactada porque

7
00:00:17.880 --> 00:00:20.280
es la sangre de la vida a cierto

... (703 more lines)

The VTT file is synchronized to the TTS assembly timeline.
Each cue shows only the current subtitle, wrapped to at most two lines.


## Playback

To play the dubbed output:

1. **Frontend:** Open <http://localhost:8501> and select the video from the list.
   The UI will load the dubbed MP4 with captions overlay.

2. **Direct file:** Play the MP4 directly from
   `pipeline_data/api/dubbed_videos/{config}/{title}.mp4` using any media player
   (e.g., VLC, mpv). Load the corresponding VTT file from `pipeline_data/api/dubbed_captions/`
   as an external subtitle track.

## Summary

The stitch stage produced:

- **Dubbed MP4** in `pipeline_data/api/dubbed_videos/{config}/` — original video with replaced audio track
- **VTT captions** in `pipeline_data/api/dubbed_captions/` — TTS-scheduled translated subtitles

Audio-only remux means no quality loss on the video track: ffmpeg copies the video
stream as-is and only replaces the audio stream with the synthesized TTS output.

For the final demo, I use config `c-e5604bc`. The stitch endpoint remuxes the original video stream with the generated Chatterbox WAV, and the VTT captions are stored as TTS-scheduled subtitles.